In [0]:
from pyspark.sql import functions as F

In [0]:
print("正在读取铜牌层物理资产...")
# 1. 坚决不再读物理 Volume，直接从元数据读取铜牌表
oi = spark.table("bronze_order_items")
print("bronze_order_items OK...")
o = spark.table("bronze_orders")
print("bronze_orders OK...")
c = spark.table("bronze_customers")
print("bronze_customers OK...")

In [0]:
silver_fact_orders = oi \
    .join(o, on='order_id', how="inner") \
    .join(c, on='customer_id', how="inner") \
    .filter(F.col("price").isNotNull()) \
    .withColumn("price", F.round(F.col("price"), 2))


##### 强行物理落盘成银牌资产表，下游做大聚合时速度会提升数百倍

In [0]:
silver_fact_orders.write.format("delta").mode("overwrite").saveAsTable("silver_fact_orders")

In [0]:
print(f"✅ 银牌层核心事实表 silver_fct_orders 物理落盘成功！当前明细行数: {silver_fact_orders.count()}")